# Optimal Portfolio Allocation with Markowitz Theory

This notebook demonstrates mean-variance portfolio optimization for eight large-cap stocks using five years of daily data.

In [ ]:
import os
import sys
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath('../src'))
from portfolio import Portfolio

np.random.seed(42)
pd.options.display.float_format = '{:.4f}'.format


In [ ]:
TICKERS = ['AAPL', 'GOOGL', 'MSFT', 'TSLA', 'JNJ', 'XOM', 'PG', 'V']
END_DATE = datetime.today().date()
START_DATE = END_DATE - timedelta(days=5 * 365)
TARGET_RETURN = 0.16  # user-adjustable annual target return
RISK_FREE_RATE = 0.03

print(f'Date range: {START_DATE} to {END_DATE}')
print('Tickers:', ', '.join(TICKERS))


In [ ]:
portfolio = Portfolio(random_seed=42)
prices = portfolio.load_data(TICKERS, str(START_DATE), str(END_DATE))
returns, mu, sigma = portfolio.compute_statistics()

prices.to_csv('../data/stock_prices.csv')
returns.to_csv('../data/daily_returns.csv')

print('Price sample:')
display(prices.tail())
print('
Expected annual returns (mu):')
display(mu.sort_values(ascending=False))
print('
Annualized covariance matrix (Sigma):')
display(sigma)


In [ ]:
corr_matrix = returns.corr()

min_var = portfolio.optimize_portfolio(method='min_variance')
max_sharpe = portfolio.optimize_portfolio(method='max_sharpe', risk_free_rate=RISK_FREE_RATE)
target_opt = portfolio.optimize_portfolio(target_return=TARGET_RETURN, method='min_variance')

print('Correlation matrix:')
display(corr_matrix)

annualized_stats = pd.DataFrame({
    'annual_return': mu,
    'annual_volatility': returns.std() * np.sqrt(252),
}).sort_values('annual_return', ascending=False)

print('
Top performers by annualized return:')
display(annualized_stats.head(3))

print('
Optimization summaries:')
print(f"Minimum Variance -> return={min_var.expected_return:.4f}, risk={min_var.volatility:.4f}, sharpe={min_var.sharpe:.4f}")
print(f"Maximum Sharpe   -> return={max_sharpe.expected_return:.4f}, risk={max_sharpe.volatility:.4f}, sharpe={max_sharpe.sharpe:.4f}")
print(f"Target Return    -> return={target_opt.expected_return:.4f}, risk={target_opt.volatility:.4f}, sharpe={target_opt.sharpe:.4f}")


In [ ]:
frontier = portfolio.compute_efficient_frontier(num_portfolios=1000)

equal_weights = np.repeat(1 / len(TICKERS), len(TICKERS))
eq_return, eq_risk = portfolio._portfolio_metrics(equal_weights)
eq_sharpe = portfolio.sharpe_ratio(equal_weights, risk_free_rate=RISK_FREE_RATE)

risk_comparison = pd.DataFrame({
    'portfolio': ['Equal Weight', 'Minimum Variance', 'Maximum Sharpe', 'Target Return'],
    'expected_return': [eq_return, min_var.expected_return, max_sharpe.expected_return, target_opt.expected_return],
    'risk': [eq_risk, min_var.volatility, max_sharpe.volatility, target_opt.volatility],
    'sharpe': [eq_sharpe, min_var.sharpe, max_sharpe.sharpe, target_opt.sharpe],
})

print('Risk/return comparison:')
display(risk_comparison)


In [ ]:
os.makedirs('../outputs', exist_ok=True)

plt.figure(figsize=(10, 6))
plt.scatter(frontier['risk'], frontier['return'], c=frontier['sharpe'], cmap='viridis', alpha=0.55, s=20)
plt.colorbar(label='Sharpe Ratio')
plt.scatter(min_var.volatility, min_var.expected_return, color='red', marker='*', s=260, label='Minimum Variance')
plt.scatter(max_sharpe.volatility, max_sharpe.expected_return, color='gold', edgecolor='black', marker='*', s=300, label='Maximum Sharpe')
plt.scatter(target_opt.volatility, target_opt.expected_return, color='blue', marker='X', s=140, label='Target Return Optimum')
plt.scatter(eq_risk, eq_return, color='black', marker='D', s=60, label='Equal Weight')
plt.title('Efficient Frontier (Random Feasible Portfolios)')
plt.xlabel('Annualized Risk (Std. Dev.)')
plt.ylabel('Annualized Expected Return')
plt.legend()
plt.tight_layout()
plt.savefig('../outputs/efficient_frontier.png', dpi=200)
plt.show()


In [ ]:
weights_df = pd.DataFrame({
    'ticker': TICKERS,
    'weight': max_sharpe.weights,
}).sort_values('weight', ascending=False)

plt.figure(figsize=(10, 5))
plt.bar(weights_df['ticker'], weights_df['weight'], color='#2a9d8f')
plt.title('Optimal Asset Weights (Maximum Sharpe Portfolio)')
plt.ylabel('Weight')
plt.xlabel('Ticker')
plt.tight_layout()
plt.savefig('../outputs/optimal_weights.png', dpi=200)
plt.show()

display(weights_df)


In [ ]:
stock_perf = pd.DataFrame({
    'ticker': TICKERS,
    'annual_return': mu.values,
    'annual_risk': (returns.std() * np.sqrt(252)).values,
})

plt.figure(figsize=(8, 6))
plt.scatter(stock_perf['annual_risk'], stock_perf['annual_return'], color='#264653', s=70)
for _, row in stock_perf.iterrows():
    plt.annotate(row['ticker'], (row['annual_risk'], row['annual_return']), xytext=(5, 4), textcoords='offset points')
plt.title('Individual Stock Risk vs Return')
plt.xlabel('Annualized Risk (Std. Dev.)')
plt.ylabel('Annualized Return')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig('../outputs/stock_risk_return.png', dpi=200)
plt.show()

display(stock_perf.sort_values('annual_return', ascending=False))


## Notes

- Set `TARGET_RETURN` to test custom risk-minimizing portfolios.
- The optimizer enforces long-only, fully invested constraints (weights sum to 1 and are non-negative).
- Results vary with market regimes and are for educational use only.